In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('stud.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
df['Percent']=(df['math_score']+df['writing_score']+df['reading_score'])//3

In [4]:
df

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,Percent
0,female,group B,bachelor's degree,standard,none,72,72,74,72
1,female,group C,some college,standard,completed,69,90,88,82
2,female,group B,master's degree,standard,none,90,95,93,92
3,male,group A,associate's degree,free/reduced,none,47,57,44,49
4,male,group C,some college,standard,none,76,78,75,76
...,...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95,94
996,male,group C,high school,free/reduced,none,62,55,55,57
997,female,group C,high school,free/reduced,completed,59,71,65,65
998,female,group D,some college,standard,completed,68,78,77,74


In [5]:
df.isnull().sum()

gender                         0
race_ethnicity                 0
parental_level_of_education    0
lunch                          0
test_preparation_course        0
math_score                     0
reading_score                  0
writing_score                  0
Percent                        0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
y=df['Percent']

In [46]:
x=df.drop(['Percent','math_score','reading_score'],axis=1)


In [47]:
x.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,writing_score
0,female,group B,bachelor's degree,standard,none,74
1,female,group C,some college,standard,completed,88
2,female,group B,master's degree,standard,none,93
3,male,group A,associate's degree,free/reduced,none,44
4,male,group C,some college,standard,none,75


In [48]:
#Create Column Transformer with 3 types of transformer
num_f=x.select_dtypes(exclude='object').columns
cat_f=x.select_dtypes(include='object').columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer= StandardScaler()
cat_transformer=OneHotEncoder()

preprocessor= ColumnTransformer(
    [
        ('OneHotEncoder', cat_transformer, cat_f),
        ('StandardScaler', numeric_transformer, num_f),
    ]
)

In [49]:
x=preprocessor.fit_transform(x)

In [50]:
x

array([[ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.        ,  0.39149181],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.        ,  1.31326868],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.        ,  1.64247471],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.        , -0.20107904],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.        ,  0.58901542],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.        ,  1.18158627]])

In [51]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=40)
x_train.shape

(800, 18)

In [52]:
from sklearn.metrics import r2_score,root_mean_squared_error,mean_absolute_percentage_error
def evaluate_model(true,pred):
    mape=mean_absolute_percentage_error(true,pred)
    r2=r2_score(true,pred)
    rmse=root_mean_squared_error(true,pred)
    return mape,r2,rmse

In [53]:
from sklearn.linear_model import Lasso
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor

In [54]:
models={
    'Lasso':Lasso(),
    'LinearRegression':LinearRegression(),
    'Ridge':Ridge(),
    'KNeighborsRegressor':KNeighborsRegressor(),
    'DecisionTreeRegressor':DecisionTreeRegressor(),
    'RandomForestRegressor':RandomForestRegressor(),
    'AdaBoostRegressor':AdaBoostRegressor(),
    'XGBRegressor':XGBRegressor()
}

model_list=[]
r2_list=[]

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)
    
    y_train_pred= model.predict(x_train)
    y_test_pred= model.predict(x_test)
    
    mape_train,r2_train,rmse_train=evaluate_model(y_train,y_train_pred)
    mape_test,r2_test,rmse_test=evaluate_model(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('.....Training set....')
    print(f' R2 Score : {r2_train}')
    print(f' Mape : {100-mape_train}')
    print(f' Rmse : {rmse_train}')
    print('---------------------')
    print('.....Test set....')
    print(f' R2 Score : {r2_test}')
    print(f' Mape : {100-mape_test}')
    print(f' Rmse : {rmse_test}')
    r2_list.append(r2_test)
    print('*'*20)
    print()
    
    
    

Lasso
.....Training set....
 R2 Score : 0.9304795837366442
 Mape : 99.95029297486815
 Rmse : 3.7758539651644827
---------------------
.....Test set....
 R2 Score : 0.9163354925777715
 Mape : 99.94716429990625
 Rmse : 4.04641942150713
********************

LinearRegression
.....Training set....
 R2 Score : 0.9710353105361845
 Mape : 99.96921149847937
 Rmse : 2.437212773541864
---------------------
.....Test set....
 R2 Score : 0.9644098331108464
 Mape : 99.96684224012763
 Rmse : 2.6391573214167243
********************

Ridge
.....Training set....
 R2 Score : 0.9711921177350733
 Mape : 99.96926108636922
 Rmse : 2.4306066064226655
---------------------
.....Test set....
 R2 Score : 0.9638021027049463
 Mape : 99.96647617380225
 Rmse : 2.6615947975683074
********************

KNeighborsRegressor
.....Training set....
 R2 Score : 0.8770164937645524
 Mape : 99.9330212059733
 Rmse : 5.022066307805981
---------------------
.....Test set....
 R2 Score : 0.8400086865712635
 Mape : 99.927913651435